# 6. Interactive Monte Carlo - Enhanced with Sliders

**Objective**: Dynamically manipulate parameters using sliders and instantly visualize their effects on both the path trajectories and final distribution.

## Interactive Controls
Use the sliders below to:
- **S0**: Adjust initial stock price ($50-$200)
- **μ (Drift)**: Change annual return expectation (-10% to +20%)
- **σ (Volatility)**: Modify price uncertainty (5%-80% annual)
- **T (Time)**: Extend simulation horizon (0.5 to 5 years)
- **N_paths**: Run more simulations for accuracy (10-1000 paths)
- **Distribution**: Compare normal vs fat-tailed (extreme event-prone) models

Watch how each parameter affects the **spread of paths** and the **final price distribution**!

In [ ]:
# Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Output
    widgets_available = True
except ImportError:
    widgets_available = False
    print("⚠ ipywidgets not available. Please install: pip install ipywidgets")

# Set matplotlib to use inline display in notebook
%matplotlib inline

In [ ]:
def plot_interactive_mc(S0, mu, sigma, T, N_paths, distribution):
    """
    Generate Monte Carlo paths and visualize with current parameters.
    
    Parameters:
    - S0: Initial stock price
    - mu: Annual drift rate (expected return)
    - sigma: Annual volatility (standard deviation of returns)
    - T: Time horizon in years
    - N_paths: Number of simulation paths
    - distribution: 'Normal' or 'Fat-Tailed'
    """
    
    # Setup simulation parameters
    N_steps = int(252 * T)  # 252 trading days per year
    dt = T / N_steps
    
    # Initialize paths array
    paths = np.zeros((N_steps, N_paths))
    paths[0] = S0
    
    # Generate Monte Carlo paths using Geometric Brownian Motion
    # dS = μ*S*dt + σ*S*dW
    for t in range(1, N_steps):
        if distribution == 'Normal':
            # Standard normal distribution (Gaussian)
            Z = np.random.standard_normal(N_paths)
        else:  # 'Fat-Tailed'
            # Student-t distribution with df=3 (more extreme events)
            Z = np.random.standard_t(3, size=N_paths)
        
        # Apply GBM formula: S(t) = S(t-1) * exp((μ - σ²/2)*dt + σ*√dt*Z)
        paths[t] = paths[t-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    # Extract final prices at T
    ST = paths[-1, :]
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # ===== LEFT PLOT: Monte Carlo Paths =====
    time_axis = np.linspace(0, T, N_steps)
    
    # Plot all paths (clean lines, no markers)
    for i in range(N_paths):
        ax1.plot(time_axis, paths[:, i], linewidth=0.6, alpha=0.4, color='steelblue')
    
    # Add mean path
    mean_path = np.mean(paths, axis=1)
    ax1.plot(time_axis, mean_path, linewidth=2.5, color='red', label='Mean Path', linestyle='-')
    
    # Add starting price reference
    ax1.axhline(S0, color='green', linestyle=':', linewidth=2, alpha=0.7, label=f'Initial: ${S0:.2f}')
    
    # Formatting
    ax1.set_title(f'{N_paths} Simulated Paths ({distribution} Noise)\n' +
                  f'S0=${S0:.0f}, μ={mu:.3f}, σ={sigma:.3f}, T={T:.1f}y',
                  fontsize=12, fontweight='bold')
    ax1.set_xlabel('Time (Years)', fontsize=11)
    ax1.set_ylabel('Stock Price ($)', fontsize=11)
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # ===== RIGHT PLOT: Final Price Distribution =====
    # Create histogram of final prices
    ax2.hist(ST, bins=50, color='skyblue', edgecolor='black', alpha=0.7, density=True)
    
    # Add statistics lines (clean, no markers)
    mean_price = ST.mean()
    median_price = np.median(ST)
    p5 = np.percentile(ST, 5)
    p95 = np.percentile(ST, 95)
    
    ax2.axvline(mean_price, color='red', linestyle='-', linewidth=2.5, label=f'Mean: ${mean_price:.2f}')
    ax2.axvline(median_price, color='orange', linestyle='--', linewidth=2.5, label=f'Median: ${median_price:.2f}')
    ax2.axvline(p5, color='purple', linestyle=':', linewidth=1.5, alpha=0.6, label=f'5th %ile: ${p5:.2f}')
    ax2.axvline(p95, color='purple', linestyle=':', linewidth=1.5, alpha=0.6, label=f'95th %ile: ${p95:.2f}')
    
    # Formatting
    ax2.set_title(f'Final Price Distribution ({distribution} Noise)\n' +
                  f'Std Dev: ${ST.std():.2f}, Skewness: {ST.std()/mean_price:.3f}',
                  fontsize=12, fontweight='bold')
    ax2.set_xlabel('Final Stock Price ($)', fontsize=11)
    ax2.set_ylabel('Probability Density', fontsize=11)
    ax2.legend(loc='best', fontsize=9)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Set x-axis limits based on data
    ax2.set_xlim(max(0, p5 - 50), p95 + 50)
    
    plt.tight_layout()
    plt.show()


# Test function
print("✓ Monte Carlo simulation function defined")

## Interactive Controls - Adjust Parameters and Watch the Graphs Update in Real-Time!

In [ ]:
if widgets_available:
    # Create interactive sliders with exact parameters from reference notebook
    # Move the sliders and observe how each affects the Monte Carlo simulation
    
    interact(plot_interactive_mc,
             S0=FloatSlider(
                 value=100, 
                 min=50, 
                 max=200, 
                 step=10,
                 description='Initial Price (S0)',
                 style={'description_width': '150px'},
                 layout={'width': '400px'}
             ),
             mu=FloatSlider(
                 value=0.08, 
                 min=-0.1, 
                 max=0.2, 
                 step=0.01,
                 description='Drift Rate (μ)',
                 style={'description_width': '150px'},
                 layout={'width': '400px'}
             ),
             sigma=FloatSlider(
                 value=0.2, 
                 min=0.05, 
                 max=0.8, 
                 step=0.05,
                 description='Volatility (σ)',
                 style={'description_width': '150px'},
                 layout={'width': '400px'}
             ),
             T=FloatSlider(
                 value=1.0, 
                 min=0.5, 
                 max=5.0, 
                 step=0.5,
                 description='Time Horizon (T)',
                 style={'description_width': '150px'},
                 layout={'width': '400px'}
             ),
             N_paths=IntSlider(
                 value=100, 
                 min=10, 
                 max=1000, 
                 step=50,
                 description='Number of Paths',
                 style={'description_width': '150px'},
                 layout={'width': '400px'}
             ),
             distribution=Dropdown(
                 options=['Normal', 'Fat-Tailed'], 
                 value='Normal',
                 description='Distribution Type',
                 style={'description_width': '150px'}
             )
    )
else:
    print("ipywidgets not available. Running static example...")
    plot_interactive_mc(100, 0.08, 0.2, 1.0, 100, 'Normal')

## Guided Exploration: Questions for Understanding

### Experiment 1: Effect of Volatility (σ)
**Question**: What happens to the cone of paths when you increase volatility from 0.2 to 0.6?
- **Observation**: The paths should spread out more, creating a wider "cone of uncertainty"
- **Why**: Higher volatility means more price swings, both up and down

### Experiment 2: Effect of Drift Rate (μ)
**Question**: How does increasing drift from 0.08 to 0.15 compare to decreasing it to -0.05?
- **Observation**: Positive drift shifts the mean final price up; negative drift shifts it down
- **Why**: Drift represents the expected return; higher μ = upward trend

### Experiment 3: Effect of Time Horizon (T)
**Question**: What happens to the distribution when you extend T from 1 year to 5 years?
- **Observation**: The distribution becomes highly skewed with a long right tail
- **Why**: Geometric Brownian Motion creates compounding; longer times magnify volatility effects

### Experiment 4: Normal vs Fat-Tailed Distribution
**Question**: Compare final distributions with T=3, σ=0.3, then toggle distribution type
- **Observation**: Fat-Tailed shows more extreme values (much higher highs, lower lows)
- **Why**: Fat-tails model real markets where crashes/rallies occur more than normal distribution predicts

### Experiment 5: Number of Paths Effect
**Question**: Compare N_paths=50 vs N_paths=500 - does the overall shape change?
- **Observation**: Shape stays similar, but rough edges smooth out with more paths
- **Why**: More simulations = better approximation of true probability distribution (Law of Large Numbers)

## Key Insights: What You're Learning

| Parameter | Range | Effect on Paths | Effect on Distribution |
|-----------|-------|-----------------|------------------------|
| **S0 (Initial Price)** | $50-$200 | Shifts all paths up/down | Shifts distribution right/left |
| **μ (Drift)** | -10% to +20% | Trends up (positive) or down (negative) | Mean moves higher/lower |
| **σ (Volatility)** | 5%-80% | Wide spread (high σ) or tight (low σ) | Wide distribution (high σ) or narrow |
| **T (Time)** | 0.5-5 years | More time = more divergence | Longer tail, higher std deviation |
| **N_paths** | 10-1000 | More paths = smoother histogram | Distribution becomes more accurate |
| **Distribution** | Normal vs Fat-Tailed | Similar appearance | Fat-tailed has extreme outliers |

### Real-World Applications
- **Risk Management**: Use the cone to estimate 5th/95th percentile (Value at Risk)
- **Option Pricing**: Monte Carlo is essential for exotic derivatives
- **Portfolio Analysis**: Simulate multi-year wealth paths to plan retirement
- **Trading Strategy**: Test strategies across thousands of potential market scenarios